In [1]:
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.fairtabdiffusion.adapter import KatabaticFairTabDiffusion # Updated Import
from utils import discretize_preprocess

# --- Configuration ---
DATASET = "adult"
raw_path = f"raw_data/{DATASET}.csv"
discretized_path = f"discretized_data/{DATASET}.csv"
output_dir = f"sample_data/{DATASET}"
synthetic_dir = f"synthetic/{DATASET}/fairtabdiffusion" # Updated Directory
real_test_dir = f"sample_data/{DATASET}"

# User Input
protected_col = input("Protected Attribute (S) [default 'sex']: ").strip() or "sex"
target_col = input("Target Attribute (Y) [default 'class']: ").strip() or "class"

# --- Model Configuration ---
model_config = {
    # FairTabDiffusion Hyperparameters
    "epochs": 100,          # Diffusion usually needs more epochs (50-100+)
    "batch_size": 256,      # Larger batch sizes often stable for diffusion
    
    # Fairness Config
    # Used by the Adapter to identify the target column for splitting artifacts
    # Used by FairnessEvaluation (if enabled) to calculate metrics
    "fairness_config": {
        "S": protected_col,
        "Y": target_col,
        "S_under": "0",
        "Y_desire": "1"
    }
}

# --- 1. Preprocess ---
print("Discretizing data...")
discretize_preprocess(
    file_path=raw_path,
    output_path=discretized_path,
    bins=10,
    strategy='uniform'
)

# --- 2. Run Pipeline ---
# Instantiate pipeline with the FairTabDiffusion Adapter class
pipeline = TrainTestSplitPipeline(model=KatabaticFairTabDiffusion)

print(f"Starting pipeline for FairTabDiffusion on {DATASET}")

pipeline.run(
    input_csv=discretized_path,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Discretizing data...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Starting pipeline for FairTabDiffusion on adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Loading FairTabDiffusion data from: sample_data/adult
Initializing FairTabDiffusion (Features:14, Epochs:100)...


Training FairTabDiffusion: 100%|██████████| 100/100 [06:50<00:00,  4.10s/it, loss=0.698]


Generating synthetic data to: synthetic/adult/fairtabdiffusion
Saved artifacts: x_synth.csv, y_synth.csv (1000 rows)


/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/fairtabdiffusion_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7620
F1 Score: 0.6667
AUC: 0.5065

MLP:
Accuracy: 0.7252
F1 Score: 0.6581
AUC: 0.4481

RF:
Accuracy: 0.7270
F1 Score: 0.6612
AUC: 0.4311

XGBoost:
Accuracy: 0.6447
F1 Score: 0.6399
AUC: 0.4735


'Train test split pipeline executed successfully.'